# Relatorio do Projeto: Reconhecimento de Sinais de Transito GTSRB

Este relatorio descreve o trabalho desenvolvido no notebook `Report.ipynb`, explicando o objetivo do projeto, os dados utilizados, as tecnicas aplicadas, os modelos testados e os principais resultados obtidos.

O projeto foi desenvolvido no ambito de Visao por Computador e Processamento de Imagem, com foco na classificacao automatica de sinais de transito usando o dataset GTSRB (*German Traffic Sign Recognition Benchmark*).



## 1. Objetivo

O objetivo principal foi construir e avaliar modelos de Deep Learning capazes de classificar imagens de sinais de transito em 43 classes diferentes.

A meta pratica do trabalho foi aproximar o desempenho de referencia do GTSRB, explorando arquiteturas de redes neuronais convolucionais, tecnicas de pre-processamento, balanceamento de dados, data augmentation, ensemble e transfer learning.

O melhor resultado reportado no desenvolvimento foi de **99.39% de accuracy no conjunto de teste**, obtido com uma arquitetura `RobustCNN` treinada a partir do zero.



## 2. Dataset Utilizado

Foi utilizado o dataset **GTSRB**, composto por imagens reais de sinais de transito capturadas em diferentes condicoes de iluminacao, escala, perspetiva, nitidez e ruido.

A organizacao usada segue o formato esperado pelo `torchvision.datasets.ImageFolder`, onde cada classe corresponde a uma pasta com o respetivo identificador numerico.

| Pasta | Funcao |
|---|---|
| `../datasets/original_train_images` | Imagens originais de treino |
| `../datasets/train_images` | Subconjunto de treino apos split |
| `../datasets/val_images` | Subconjunto de validacao |
| `../datasets/train_balanced` | Treino balanceado com augmentation estatica |
| `../datasets/test_images` | Conjunto de teste final |



## 3. Bibliotecas e Ferramentas

O projeto foi implementado em Python, usando principalmente:

| Biblioteca | Utilizacao |
|---|---|
| `torch` | Definicao, treino e avaliacao das redes neuronais |
| `torchvision` | Transforms, datasets, modelos pre-treinados e augmentations |
| `numpy` | Operacoes numericas auxiliares |
| `matplotlib` | Visualizacao de imagens, curvas de treino e graficos |
| `pandas` | Organizacao de matrizes e tabelas |
| `seaborn` | Visualizacao de matrizes de confusao |
| `sklearn.metrics` | Calculo da matriz de confusao |
| `OpenCV` / `cv2` | Tecnicas classicas de processamento de imagem, como CLAHE |
| `PIL` | Leitura e manipulacao de imagens |
| `torchinfo` | Inspecao das arquiteturas |

Foram tambem usados ficheiros utilitarios locais: `util/vcpi_util.py` e `util/our_util.py`.



## 4. Preparacao dos Dados

A preparacao dos dados incluiu quatro passos principais:

1. Carregamento das imagens com `datasets.ImageFolder`.
2. Redimensionamento das imagens para uma resolucao fixa, inicialmente `32x32`.
3. Conversao para tensor com `transforms.ToTensor()`.
4. Separacao entre treino, validacao e teste.

Foi dada atencao especial ao conjunto de validacao, porque o GTSRB contem imagens sequenciais. Imagens muito parecidas podem aparecer em frames proximas, por isso o split deve evitar que sequencias semelhantes fiquem ao mesmo tempo no treino e na validacao.



## 5. Analise Exploratoria

Antes do treino, o notebook faz uma analise visual e estatistica do dataset:

- visualizacao de amostras aleatorias de treino, validacao e teste;
- verificacao da distribuicao das classes;
- identificacao do desbalanceamento do dataset.

Esta etapa foi importante porque o GTSRB nao tem uma distribuicao uniforme. Sem correcao, o modelo poderia aprender melhor as classes maioritarias e ter pior desempenho nas classes raras.



## 6. Balanceamento e Data Augmentation

Para lidar com o desbalanceamento, foi criado um conjunto `train_balanced`, onde classes minoritarias foram aumentadas ate um numero alvo de imagens.

Foram usadas tecnicas de augmentation estatica, isto e, novas imagens foram criadas e gravadas em disco antes do treino.

Transformacoes usadas ou testadas:

- rotacoes pequenas;
- translacoes;
- alteracao de escala;
- shear;
- alteracao de brilho, contraste e saturacao;
- perspetiva;
- blur;
- ruido suave;
- sharpness e grayscale como opcoes experimentais.

Tambem foram testadas estrategias de **dynamic augmentation**, aplicadas durante o treino com `torchvision.transforms`.



## 7. Modelo Baseline

O primeiro modelo implementado foi uma CNN simples, usada como referencia experimental.

A `BaselineCNN` contem:

- duas camadas convolucionais;
- ativacoes `ReLU`;
- `MaxPool2d` para reducao espacial;
- uma camada totalmente ligada final.

Este modelo serviu para estabelecer um ponto de partida. O desempenho reportado para o baseline foi aproximadamente **87.12% no test set** na conclusao consolidada do notebook.



## 8. Infraestrutura de Treino

O treino foi feito com uma funcao generica `train_model`, responsavel por:

- colocar o modelo no device correto (`CPU` ou `GPU`);
- executar forward pass e backward pass;
- atualizar os pesos com o otimizador;
- calcular loss e accuracy de treino;
- avaliar em validacao apos cada epoca;
- aplicar scheduler de learning rate;
- guardar automaticamente o melhor checkpoint com base na validation loss;
- parar cedo com `Early_Stopping` quando a validacao deixa de melhorar.

Foram usados `CrossEntropyLoss`, `Adam`, `ReduceLROnPlateau` e `Early_Stopping`.



## 9. RobustCNN

A principal melhoria do projeto foi a substituicao da baseline por uma arquitetura mais robusta, chamada `RobustCNN`.

A arquitetura inclui:

- tres blocos convolucionais;
- maior numero de canais (`32`, `64`, `128`);
- `BatchNorm2d` apos convolucoes;
- ativacoes `ReLU`;
- `MaxPool2d` para reducao espacial;
- `Dropout` para regularizacao;
- **Global Average Pooling**, reduzindo a dependencia de uma camada `Flatten` grande.

Esta alteracao foi a que trouxe o maior ganho do projeto. Segundo a conclusao do notebook, a passagem de `BaselineCNN` para `RobustCNN` elevou o resultado de **87.12% para 99.39%** no test set.



## 10. Experiencias com Augmentation Dinamica

Depois da `RobustCNN`, foram testadas varias variantes de augmentation dinamica:

| Estrategia | Ideia |
|---|---|
| Static only | Treino com dataset ja balanceado em disco |
| Static + dynamic | Dataset balanceado + transforms aleatorias durante treino |
| Dynamic only | Dataset com augmentation em tempo de treino sem aumentar ficheiros |
| Massive dynamic | Concatenacao de multiplas versoes dinamicas do dataset |

Apesar de algumas destas estrategias melhorarem a validation accuracy, elas nao superaram de forma consistente a `RobustCNN` estatica no test set.



## 11. Processamento Classico de Imagem

Tambem foram testadas tecnicas classicas de processamento de imagem para tentar melhorar a robustez visual dos sinais.

Foram exploradas:

- **CLAHE** no canal de luminancia em espaco LAB;
- normalizacao por media e desvio-padrao do dataset;
- gamma correction para simular variacoes de iluminacao;
- combinacao de CLAHE com augmentation dinamica.

Estas tecnicas sao teoricamente adequadas para lidar com sombras e iluminacao irregular. No entanto, no notebook, as experiencias com CLAHE e normalizacao nao superaram claramente o pipeline mais simples com `RobustCNN`.



## 12. Experiencia com Resolucao

O projeto tambem testou a diferenca entre imagens `32x32` e `48x48`.

A hipotese era que `48x48` poderia preservar mais detalhe visual dos sinais. Contudo, aumentar a resolucao tambem aumenta o custo computacional e pode exigir mais regularizacao.

No desenvolvimento registado, a resolucao maior nao produziu uma melhoria clara suficiente para substituir o melhor resultado obtido com o setup mais simples.



## 13. Ensembles

Foram testados ensembles com varios checkpoints treinados em condicoes diferentes.

Duas estrategias foram usadas:

- **Soft voting**: soma dos logits antes do `argmax`.
- **Hard voting**: votacao por classe prevista.

O objetivo era verificar se modelos treinados com pipelines diferentes cometiam erros complementares. No entanto, o ensemble nao superou claramente a melhor `RobustCNN` individual.



## 14. Transfer Learning

Foram tambem testados modelos pre-treinados no ImageNet:

- `ResNet-18`;
- `EfficientNet-B0`.

A estrategia incluiu uma fase inicial de treino apenas da cabeca de classificacao e depois fine-tuning completo.

Apesar de serem arquiteturas fortes, os resultados reportados ficaram abaixo da melhor `RobustCNN` treinada do zero. Isto e plausivel porque o GTSRB e um problema especifico, com imagens pequenas e classes bem definidas.



## 15. Test-Time Augmentation

Foi adicionada uma melhoria adicional ao notebook: **Test-Time Augmentation (TTA)**.

A TTA nao altera o treino nem os pesos do modelo. Durante a avaliacao, cada imagem de teste e avaliada em varias versoes ligeiramente modificadas, e os logits sao combinados antes da decisao final.

As transformacoes usadas foram conservadoras:

- brilho ligeiramente maior;
- brilho ligeiramente menor;
- contraste leve;
- deslocamentos de 1 pixel em quatro direcoes.

Nao foram usados flips horizontais porque, em sinais de transito, uma inversao pode mudar o significado visual ou criar uma imagem pouco realista.



## 16. Resultados Principais

| Modelo / Estrategia | Test Accuracy reportada |
|---|---:|
| BaselineCNN | 87.12% |
| RobustCNN static | **99.39%** |
| RobustCNN static + dynamic | 99.00% |
| RobustCNN dynamic only | 99.07% |
| RobustCNN massive dynamic | 98.80% |
| CLAHE + normalization | 97.51% |
| CLAHE + dynamic | 98.82% |
| ResNet-18 Transfer Learning | 98.46% |
| EfficientNet-B0 Transfer Learning | 98.24% |
| Ensemble soft voting | ~99.22% - 99.28% |
| Ensemble hard voting | ~99.22% - 99.25% |

O melhor resultado identificado foi a `RobustCNN` treinada a partir do zero com dados estaticamente balanceados.



## 17. Interpretacao dos Resultados

A conclusao principal e que a arquitetura teve mais impacto do que as restantes tecnicas.

A `BaselineCNN` tinha capacidade limitada, o que se refletiu num desempenho claramente inferior. A `RobustCNN`, ao adicionar mais profundidade, batch normalization, dropout e global average pooling, conseguiu generalizar muito melhor.

As tecnicas adicionais, como augmentation dinamica, CLAHE, ensembles e transfer learning, foram uteis para exploracao experimental, mas nao ultrapassaram consistentemente o modelo principal.



## 18. Pontos Fortes do Trabalho

O trabalho apresenta varios aspetos positivos:

- pipeline completo de treino, validacao e teste;
- analise da distribuicao das classes;
- criacao de dataset balanceado;
- comparacao entre baseline e modelo melhorado;
- uso de early stopping e checkpointing;
- avaliacao com matrizes de confusao;
- comparacao com transfer learning;
- exploracao de tecnicas classicas de processamento de imagem;
- teste de ensembles;
- adicao de TTA para avaliacao mais robusta.



## 19. Limitacoes e Cuidados

Existem alguns pontos que devem ser tratados com cuidado numa entrega final:

1. Algumas celulas do notebook contem outputs antigos ou resultados de execucoes diferentes. Para uma entrega final, o ideal e fazer uma execucao limpa do notebook do inicio ao fim.
2. Alguns valores aparecem escritos manualmente em tabelas finais. O mais rigoroso seria calcular todos os resultados diretamente a partir das predicoes guardadas.
3. A comparacao entre modelos deve garantir que todos usam exatamente o mesmo test set e transforms equivalentes.
4. Para resultados muito altos, pequenas diferencas de seed, split ou checkpoint podem alterar a accuracy final.
5. A accuracy global deve ser complementada com metricas por classe.



## 20. Melhorias Futuras

Possiveis melhorias futuras:

- calcular precision, recall e F1-score por classe;
- guardar automaticamente os resultados de cada experiencia em CSV;
- adicionar seeds fixas para maior reprodutibilidade;
- testar `AdamW` com `CosineAnnealingLR`;
- testar `label_smoothing` na `CrossEntropyLoss`;
- fazer weighted ensemble apenas com os melhores modelos;
- analisar visualmente todas as imagens mal classificadas;
- comparar resultados com e sem TTA numa execucao limpa;
- guardar o epoch e a validation loss do melhor checkpoint.



## 21. Conclusao

O projeto desenvolveu um pipeline completo para classificacao de sinais de transito no dataset GTSRB. A evolucao principal foi a passagem de uma CNN simples para uma arquitetura `RobustCNN`, que aumentou significativamente a capacidade de representacao e reduziu overfitting atraves de batch normalization, dropout e global average pooling.

O melhor resultado reportado foi **99.39% de accuracy no conjunto de teste**, valor bastante elevado para o problema. As experiencias adicionais com augmentation, CLAHE, ensembles e transfer learning contribuiram para validar escolhas de projeto, mesmo quando nao superaram o melhor modelo.

Assim, o trabalho demonstra nao so a construcao de um modelo de alto desempenho, mas tambem uma abordagem experimental comparativa, onde diferentes tecnicas foram testadas e avaliadas de forma critica.

